### 1.读取数据

In [10]:
import pandas as pd

output_path='../Data/extracted/2025/07/01/2025-07-01-1.json'
df= pd.read_json(output_path, lines=True)

In [11]:
df.head(5)  # Display the first 10 rows of the DataFrame

,id,type,actor,repo,payload,public,created_at,org
0,51530838144,PushEvent,"{'id': 158555496, 'login': 'VitalikK5', 'displ...","{'id': 998567711, 'name': 'VitalikK5/online-bo...","{'repository_id': 998567711, 'push_id': 252164...",True,2025-07-01 01:00:00+00:00,NaN
1,51530838154,PushEvent,"{'id': 1415420, 'login': 'kaapow', 'display_lo...","{'id': 646000227, 'name': 'kaapow/Journal', 'u...","{'repository_id': 646000227, 'push_id': 252164...",True,2025-07-01 01:00:00+00:00,NaN
2,51530838158,PushEvent,"{'id': 41898282, 'login': 'github-actions[bot]...","{'id': 543673983, 'name': 'LucaLSN/LucaLSN', '...","{'repository_id': 543673983, 'push_id': 252164...",True,2025-07-01 01:00:00+00:00,NaN
3,51530838167,PushEvent,"{'id': 44527179, 'login': 'xsrazy', 'display_l...","{'id': 1010067851, 'name': 'xsrazy/GitHub-Auto...","{'repository_id': 1010067851, 'push_id': 25216...",True,2025-07-01 01:00:00+00:00,NaN
4,51530838168,IssueCommentEvent,"{'id': 58986931, 'login': 'lkingland', 'displa...","{'id': 242137258, 'name': 'knative/func', 'url...","{'action': 'created', 'issue': {'url': 'https:...",True,2025-07-01 01:00:00+00:00,"{'id': 35583233, 'login': 'knative', 'gravatar..."


In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 153740 entries, 0 to 153739
Data columns (total 8 columns):
 #   Column      Non-Null Count   Dtype              
---  ------      --------------   -----              
 0   id          153740 non-null  int64              
 1   type        153740 non-null  object             
 2   actor       153740 non-null  object             
 3   repo        153740 non-null  object             
 4   payload     153740 non-null  object             
 5   public      153740 non-null  bool               
 6   created_at  153740 non-null  datetime64[ns, UTC]
 7   org         32403 non-null   object             
dtypes: bool(1), datetime64[ns, UTC](1), int64(1), object(5)
memory usage: 8.4+ MB


In [13]:
df.columns

Index(['id', 'type', 'actor', 'repo', 'payload', 'public', 'created_at',
       'org'],
      dtype='object')

### 2.分离数据

这里将数据分离为 三类节点：这里用三张表来记录节点以及属性
1. 开发者（Actor Node）：
    - actor_id: 唯一标识符  
    - actor_login:
    - created_at: 
2. 仓库节点（Repo Node）：
    - repo_id: 唯一标识符
    - repo_name: 仓库名称
    - eco_name: 生态系统名称
    - is_abnormal
    - created_at
    - custom_marks
    - upstream_marks
3. 组织节点（Org Node）：
    - org_id: 唯一标识符
    - org_login

在用三张表来记录节点之间的关系：
1. 开发者与仓库（Event）：Developer → Repository
    - event_type：
    - payload：
    - body：
    - created_at：
    - public：
2. 仓库与仓库（Upstream）：Repository → Repository
3. 开发者与组织（Belong）：Repository → Organization


   



#### 2.1 分离开发者节点（Actor Node）

1.  遍历 获取actor节点信息


In [ ]:
actor_nodes = {}

# 遍历数据
for _, row in df.iterrows():
    #  actor
    actor = row.get('actor', {})
    actor_id = actor.get('id')
    if pd.notnull(actor_id):
        actor_nodes[actor_id] = {
            'id': actor_id,
            'actor_name': actor.get('login'),
        }
    
    # payload.issue.user
    payload = row.get('payload', {})
    issue = payload.get('issue', {})
    issue_user = issue.get('user', {})
    user_id = issue_user.get('id')
    if pd.notnull(user_id):
        actor_nodes[user_id] = {
            'id': user_id,
            'actor_name': issue_user.get('login'),
        }

# 转为 DataFrame
actor_df = pd.DataFrame(list(actor_nodes.values()))

# 按 id 排序（可选）
actor_df = actor_df.sort_values(by='id').reset_index(drop=True)

In [17]:
actor_df.head(5)  # Display the first 5 rows of the actor DataFrame

,id,login,avatar_url,gravatar_id,url
0,4,wycats,https://avatars.githubusercontent.com/u/4?,,https://api.github.com/users/wycats
1,76,rabble,https://avatars.githubusercontent.com/u/76?,,https://api.github.com/users/rabble
2,217,tkersey,https://avatars.githubusercontent.com/u/217?,,https://api.github.com/users/tkersey
3,331,vic,https://avatars.githubusercontent.com/u/331?,,https://api.github.com/users/vic
4,530,fauxparse,https://avatars.githubusercontent.com/u/530?,,https://api.github.com/users/fauxparse
